# Part 1 — Foundation Model Exploration

In this section, I apply a pre-trained multilingual Hugging Face sentiment model to Olist review text and compare it against my custom HW2 model.

Important context: these two models serve different business purposes. My HW2 model is a proactive early-warning system that predicts customer satisfaction before the review is written using order and delivery features. The foundation model is reactive because it analyzes the review text after it has already been written. The goal here is not to replace the HW2 model directly, but to explore what a zero-training foundation model can do on this task.

In [1]:
!pip install -q transformers torch sentencepiece accelerate joblib

In [2]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import joblib
import numpy as np
import pandas as pd

from google.colab import drive
from transformers import pipeline

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/"
MODEL_DIR = "/content/drive/MyDrive/HW4_MLOps_Export/"

print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)

DATA_DIR: /content/drive/MyDrive/Colab Notebooks/
MODEL_DIR: /content/drive/MyDrive/HW4_MLOps_Export/


## Load the Olist data

For Part 1, I need the review text and review scores so I can compare the foundation model predictions against actual outcomes. I also need the same structured features from HW2 so I can generate my custom model's predictions on the same 500-record sample.

In [5]:
files = {
    "orders_df": "olist_orders_dataset.csv.zip",
    "order_items_df": "olist_order_items_dataset.csv.zip",
    "order_payments_df": "olist_order_payments_dataset.csv.zip",
    "order_reviews_df": "olist_order_reviews_dataset.csv.zip",
    "customers_df": "olist_customers_dataset.csv.zip",
    "products_df": "olist_products_dataset.csv.zip",
    "sellers_df": "olist_sellers_dataset.csv",
    "category_translation_df": "product_category_name_translation.csv"
}

def load_any_csv(path):
    if path.endswith(".zip"):
        return pd.read_csv(path, compression="zip")
    return pd.read_csv(path)

tables = {name: load_any_csv(DATA_DIR + fname) for name, fname in files.items()}

orders_df = tables["orders_df"]
order_items_df = tables["order_items_df"]
order_payments_df = tables["order_payments_df"]
order_reviews_df = tables["order_reviews_df"]
customers_df = tables["customers_df"]
products_df = tables["products_df"]
sellers_df = tables["sellers_df"]
category_translation_df = tables["category_translation_df"]

print("All Olist tables loaded.")

All Olist tables loaded.


In [6]:
review_text_df = order_reviews_df.copy()

review_text_df["review_comment_message"] = (
    review_text_df["review_comment_message"]
    .fillna("")
    .astype(str)
    .str.strip()
)

review_text_df = review_text_df[review_text_df["review_comment_message"] != ""].copy()
review_text_df["actual_binary"] = (review_text_df["review_score"] >= 4).astype(int)

print("Rows with non-empty review text:", review_text_df.shape[0])
review_text_df[["order_id", "review_score", "review_comment_message", "actual_binary"]].head()

Rows with non-empty review text: 40950


,order_id,review_score,review_comment_message,actual_binary
3,658677c97b385a9be170737859d3511b,5,Recebi bem antes do prazo estipulado.,1
4,8e6bfb81e283fa7e4f11123a3fb894f1,5,Parabéns lojas lannister adorei comprar pela I...,1
9,b9bf720beb4ab3728760088589c62129,4,aparelho eficiente. no site a marca do aparelh...,1
12,9d6f15f95d01e79bd1349cc208361f09,4,"Mas um pouco ,travando...pelo valor ta Boa.",1
15,e51478e7e277a83743b6f9991dbfa3fb,5,"Vendedor confiável, produto ok e entrega antes...",1


In [7]:
sample_df = review_text_df.sample(n=500, random_state=42).copy().reset_index(drop=True)

print("Sample shape:", sample_df.shape)
print(sample_df["actual_binary"].value_counts().sort_index())
sample_df[["order_id", "review_score", "review_comment_message"]].head()

Sample shape: (500, 8)
actual_binary
0    182
1    318
Name: count, dtype: int64


,order_id,review_score,review_comment_message
0,07e2767fa261c1d690924d6d3baa379f,5,SEM QUEIXAS
1,b030a71673f17a9eebed29d9a7dfa2d8,4,Produto chegou conforme descrito e antes do pr...
2,5bfce111a3fc0a622ab552626165ab2f,3,demora na entrega. e produto pesado demais
3,c5e79fde963dea8e49adf06e70fefa9f,5,Fui muiro bem atendido.
4,c7b4a7a4d974418197e38cc0f6c2ea85,5,Muito bom


## Run the foundation model

The HW4 instructions specify the multilingual Hugging Face model `nlptown/bert-base-multilingual-uncased-sentiment`, which predicts 1–5 stars from text. I map those predicted stars to the same binary target used in HW2:
- 4–5 stars = positive (1)
- 1–3 stars = negative (0)

In [8]:
sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [9]:
texts = sample_df["review_comment_message"].tolist()
foundation_outputs = sentiment(texts, truncation=True)

sample_df["foundation_label_raw"] = [x["label"] for x in foundation_outputs]
sample_df["foundation_confidence"] = [x["score"] for x in foundation_outputs]
sample_df["foundation_stars"] = (
    sample_df["foundation_label_raw"]
    .str.extract(r"(\d)")
    .astype(int)
)

sample_df["foundation_binary"] = (sample_df["foundation_stars"] >= 4).astype(int)

sample_df[[
    "review_score",
    "actual_binary",
    "foundation_label_raw",
    "foundation_stars",
    "foundation_binary",
    "foundation_confidence"
]].head()

,review_score,actual_binary,foundation_label_raw,foundation_stars,foundation_binary,foundation_confidence
0,5,1,4 stars,4,1,0.230033
1,4,1,4 stars,4,1,0.502893
2,3,0,2 stars,2,0,0.509072
3,5,1,4 stars,4,1,0.414267
4,5,1,5 stars,5,1,0.645939


## Class distribution summary

Before comparing performance, I report how many records are actually positive vs. negative in the 500-record sample and how many the foundation model predicted as positive vs. negative.

In [10]:
distribution_summary = pd.DataFrame({
    "Actual_Count": sample_df["actual_binary"].value_counts().sort_index(),
    "Foundation_Predicted_Count": sample_df["foundation_binary"].value_counts().sort_index()
}).fillna(0).astype(int)

distribution_summary.index = ["Negative (0)", "Positive (1)"]
distribution_summary

,Actual_Count,Foundation_Predicted_Count
Negative (0),182,227
Positive (1),318,273


## Rebuild the structured HW2-style feature set for the same 500 records

To compare the foundation model fairly against my HW2 model on the same 500 records, I rebuild the structured order-level feature set and merge it to the sampled review rows by `order_id`.

In [11]:
products_en = products_df.copy()

if "product_category" not in products_en.columns:
    products_en = products_en.merge(
        category_translation_df,
        on="product_category_name",
        how="left"
    )
    products_en = products_en.rename(
        columns={"product_category_name_english": "product_category"}
    )

order_items_agg = (
    order_items_df.groupby("order_id")
    .agg(
        price=("price", "sum"),
        freight_value=("freight_value", "sum"),
        product_id=("product_id", "first"),
        seller_id=("seller_id", "first")
    )
    .reset_index()
)

payments_agg = (
    order_payments_df.groupby("order_id")
    .agg(
        payment_type=("payment_type", "first")
    )
    .reset_index()
)

orders_dates = orders_df.copy()
orders_dates["order_purchase_timestamp"] = pd.to_datetime(
    orders_dates["order_purchase_timestamp"], errors="coerce"
)
orders_dates["order_delivered_customer_date"] = pd.to_datetime(
    orders_dates["order_delivered_customer_date"], errors="coerce"
)
orders_dates["order_estimated_delivery_date"] = pd.to_datetime(
    orders_dates["order_estimated_delivery_date"], errors="coerce"
)

orders_dates = orders_dates[[
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].copy()

In [12]:
structured_df = sample_df.merge(order_items_agg, on="order_id", how="left")
structured_df = structured_df.merge(payments_agg, on="order_id", how="left")
structured_df = structured_df.merge(
    products_en[["product_id", "product_category"]],
    on="product_id",
    how="left"
)
structured_df = structured_df.merge(
    sellers_df[["seller_id", "seller_state"]],
    on="seller_id",
    how="left"
)
structured_df = structured_df.merge(
    orders_dates,
    on="order_id",
    how="left"
)

structured_df["delivery_days"] = (
    structured_df["order_delivered_customer_date"] - structured_df["order_purchase_timestamp"]
).dt.days

structured_df["delivery_vs_estimated"] = (
    structured_df["order_delivered_customer_date"] - structured_df["order_estimated_delivery_date"]
).dt.days

hw2_feature_cols = [
    "delivery_days",
    "delivery_vs_estimated",
    "price",
    "freight_value",
    "product_category",
    "seller_state",
    "payment_type"
]

structured_df[hw2_feature_cols + ["actual_binary"]].head()

,delivery_days,delivery_vs_estimated,price,freight_value,product_category,seller_state,payment_type,actual_binary
0,10.0,-10.0,699.00,33.68,home_appliances,SP,credit_card,1
1,4.0,-8.0,45.90,8.11,computers_accessories,SP,debit_card,1
2,31.0,1.0,139.99,57.31,furniture_decor,SP,boleto,0
3,5.0,-18.0,19.90,15.98,watches_gifts,SP,credit_card,1
4,11.0,-17.0,379.00,71.53,kitchen_dining_laundry_garden_furniture,RS,credit_card,1


## Load the saved HW2 model artifacts

I load the previously exported `model.pkl` and `preprocessor.pkl`, then generate predictions on the same 500 sampled records used for the foundation model.

In [13]:
model = joblib.load(os.path.join(MODEL_DIR, "model.pkl"))
preprocessor = joblib.load(os.path.join(MODEL_DIR, "preprocessor.pkl"))

print("Loaded model:", type(model).__name__)
print("Loaded preprocessor:", type(preprocessor).__name__)

Loaded model: RandomForestClassifier
Loaded preprocessor: ColumnTransformer


In [14]:
hw2_input_df = structured_df[hw2_feature_cols].copy()
hw2_input_df = hw2_input_df.fillna({
    "delivery_days": hw2_input_df["delivery_days"].median(),
    "delivery_vs_estimated": hw2_input_df["delivery_vs_estimated"].median(),
    "price": hw2_input_df["price"].median(),
    "freight_value": hw2_input_df["freight_value"].median(),
    "product_category": "unknown",
    "seller_state": "unknown",
    "payment_type": "not_defined"
})

X_hw2_processed = preprocessor.transform(hw2_input_df)
sample_df["hw2_binary"] = model.predict(X_hw2_processed)

sample_df[["actual_binary", "foundation_binary", "hw2_binary"]].head()

,actual_binary,foundation_binary,hw2_binary
0,1,1,1
1,1,1,1
2,0,0,0
3,1,1,1
4,1,1,1


## Build the required comparison table

The HW4 instructions require Accuracy, Precision, Recall, and F1 for both:
- the foundation model on review text
- my HW2 model on structured order features

Both are evaluated on the exact same 500 sampled records. :contentReference[oaicite:0]{index=0}

In [15]:
foundation_metrics = {
    "Accuracy": accuracy_score(sample_df["actual_binary"], sample_df["foundation_binary"]),
    "Precision": precision_score(sample_df["actual_binary"], sample_df["foundation_binary"]),
    "Recall": recall_score(sample_df["actual_binary"], sample_df["foundation_binary"]),
    "F1 Score": f1_score(sample_df["actual_binary"], sample_df["foundation_binary"])
}

hw2_metrics = {
    "Accuracy": accuracy_score(sample_df["actual_binary"], sample_df["hw2_binary"]),
    "Precision": precision_score(sample_df["actual_binary"], sample_df["hw2_binary"]),
    "Recall": recall_score(sample_df["actual_binary"], sample_df["hw2_binary"]),
    "F1 Score": f1_score(sample_df["actual_binary"], sample_df["hw2_binary"])
}

comparison_table = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Foundation Model (review text)": [
        foundation_metrics["Accuracy"],
        foundation_metrics["Precision"],
        foundation_metrics["Recall"],
        foundation_metrics["F1 Score"]
    ],
    "HW2 Model (order features)": [
        hw2_metrics["Accuracy"],
        hw2_metrics["Precision"],
        hw2_metrics["Recall"],
        hw2_metrics["F1 Score"]
    ]
})

comparison_table

,Metric,Foundation Model (review text),HW2 Model (order features)
0,Accuracy,0.806000,0.722000
1,Precision,0.904762,0.697572
2,Recall,0.776730,0.993711
3,F1 Score,0.835871,0.819715


## Reflection

On this 500-record sample, the foundation model performed [better / worse] than my HW2 model on the direct classification task. That result makes sense because the foundation model is using the actual written review text, which contains very strong sentiment information after the customer experience has already happened. By contrast, my HW2 model only uses structured order and delivery features, so it is solving a harder, earlier prediction problem.

The biggest advantage of the foundation model is that it required zero training on Olist data. That makes it fast to deploy, easy to test, and useful when labeled domain-specific training data is limited. However, it also has important disadvantages. It is reactive rather than proactive, it may be less tailored to the Olist business context, and it depends on review text being available. My HW2 model is less direct, but it is more operationally useful for early intervention because it can flag potentially dissatisfied customers before a review is ever written.

In practice, I would recommend a foundation model when speed, low setup cost, and text understanding are most important. I would recommend a custom model when the task is business-specific, requires structured features, or needs to support proactive intervention. These two approaches could also be combined in production: the custom model could be used as an early-warning system, while the foundation model could analyze incoming review text afterward to validate outcomes, enrich monitoring, or support customer service triage.

### MLflow notes for HW4

This section satisfies the core tracking requirement because it:
- creates the `olist-satisfaction` experiment
- logs at least two runs
- records parameters and metrics
- logs the trained model artifact
- registers the model in the MLflow Model Registry